# 足球转会市场的网络与地理分析

本 Notebook 基于真实转会费记录，分析足球人才流动的网络结构、地理距离、区域路径和球员标签。
核心问题：足球俱乐部如何通过转会支出揭示市场偏好？全球足球人才如何在俱乐部、联赛和国家之间流动？

In [1]:
from pathlib import Path
import math
import os
import re
import unicodedata
import warnings
from collections import Counter, defaultdict

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib_cache").resolve()))
os.environ.setdefault("MPLBACKEND", "Agg")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path as MplPath
import seaborn as sns
import networkx as nx
from wordcloud import WordCloud

RANDOM_STATE = 42
BASE_DIR = Path(".")
CHART_DIR = BASE_DIR / "charts"
OUTPUT_DIR = BASE_DIR / "outputs"
CACHE_DIR = BASE_DIR / ".cache"
for directory in [CHART_DIR, OUTPUT_DIR, CACHE_DIR]:
  directory.mkdir(exist_ok=True)

TRANSFER_CSV = BASE_DIR / "Transfer Market Data.csv"
PLAYERS_CSV = BASE_DIR / "male_players.csv"
TEAMS_CSV = BASE_DIR / "male_teams.csv"

BG = "#0B1020"
AX_BG = "#111827"
GRID = "#2A3348"
TEXT = "#FFFFFF"
MUTED = "#CBD5E1"
CYAN = "#22D3EE"
GOLD = "#FBBF24"
PINK = "#F472B6"
GREEN = "#34D399"
ORANGE = "#FB923C"
PURPLE = "#A78BFA"
RED = "#F87171"
BLUE = "#60A5FA"

plt.style.use("dark_background")
plt.rcParams.update({
  "figure.facecolor": BG,
  "axes.facecolor": AX_BG,
  "savefig.facecolor": BG,
  "axes.edgecolor": GRID,
  "axes.labelcolor": TEXT,
  "xtick.color": TEXT,
  "ytick.color": TEXT,
  "text.color": TEXT,
  "legend.facecolor": AX_BG,
  "legend.edgecolor": GRID,
  "legend.labelcolor": TEXT,
  "axes.titleweight": "bold",
  "axes.titlesize": 16,
  "axes.labelsize": 11,
  "font.sans-serif": ["Arial Unicode MS", "SimHei", "DejaVu Sans"],
  "axes.unicode_minus": False,
  "figure.dpi": 120,
})
sns.set_theme(style="darkgrid", rc={
  "figure.facecolor": BG,
  "axes.facecolor": AX_BG,
  "grid.color": GRID,
  "axes.edgecolor": GRID,
})

PALETTE = [CYAN, GOLD, PINK, GREEN, ORANGE, PURPLE, BLUE, RED]

def savefig(name):
  path = CHART_DIR / name
  plt.savefig(path, dpi=220, bbox_inches="tight", facecolor=BG)
  plt.close()
  print(f"Saved: {path}")

def style_axis(ax, title=None, subtitle=None):
  ax.set_facecolor(AX_BG)
  ax.grid(True, color=GRID, alpha=0.32)
  for spine in ax.spines.values():
    spine.set_color(GRID)
  ax.tick_params(colors=TEXT)
  ax.xaxis.label.set_color(TEXT)
  ax.yaxis.label.set_color(TEXT)
  if title:
    ax.set_title(title, loc="left", pad=16, color=TEXT, fontsize=17, fontweight="bold")
  if subtitle:
    ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=MUTED, fontsize=10.5, ha="left")
  return ax

def euro_m(value):
  return value / 1_000_000

def normalize_name(value):
  if pd.isna(value):
    return ""
  text = unicodedata.normalize("NFKD", str(value))
  text = "".join(ch for ch in text if not unicodedata.combining(ch))
  text = re.sub(r"[^a-zA-Z0-9 ]+", " ", text.lower())
  return re.sub(r"\s+", " ", text).strip()

def add_labels(ax, data, x, y, label, n=8, dx=0.01, dy=0.01):
  top = data.sort_values(y, ascending=False).head(n)
  for _, row in top.iterrows():
    ax.text(row[x] + dx, row[y] + dy, str(row[label]), fontsize=8.5, color=TEXT,
            path_effects=[pe.withStroke(linewidth=3, foreground=BG)])

def clean_label(label, max_len=24):
  text = str(label)
  return text if len(text) <= max_len else text[:max_len - 1] + "…"

def haversine_km(lat1, lon1, lat2, lon2):
  lat1 = np.radians(lat1.astype(float))
  lon1 = np.radians(lon1.astype(float))
  lat2 = np.radians(lat2.astype(float))
  lon2 = np.radians(lon2.astype(float))
  dlat = lat2 - lat1
  dlon = lon2 - lon1
  a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
  return 6371.0 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def draw_curve(ax, x0, y0, x1, y1, color, width, alpha=0.55, curve=0.22):
  ctrl_x = (x0 + x1) / 2
  ctrl_y = (y0 + y1) / 2 + curve * abs(x1 - x0)
  path = MplPath([(x0, y0), (ctrl_x, ctrl_y), (x1, y1)], [MplPath.MOVETO, MplPath.CURVE3, MplPath.CURVE3])
  patch = PathPatch(path, facecolor="none", edgecolor=color, lw=width, alpha=alpha, capstyle="round")
  ax.add_patch(patch)
  return patch

def radial_layout(nodes, scores):
  ordered = sorted(nodes, key=lambda n: scores.get(n, 0), reverse=True)
  pos = {}
  for i, node in enumerate(ordered):
    ring = 0 if i < 8 else 1 if i < 28 else 2
    start = 0 if ring == 0 else 8 if ring == 1 else 28
    size = 8 if ring == 0 else 20 if ring == 1 else max(1, len(ordered) - 28)
    angle = 2 * np.pi * (i - start) / size
    radius = 0.28 if ring == 0 else 0.62 if ring == 1 else 1.0
    pos[node] = np.array([radius * np.cos(angle), radius * np.sin(angle)])
  return pos

## 1. 环境设置与数据加载

In [2]:
transfers_raw = pd.read_csv(TRANSFER_CSV)
transfers = transfers_raw.copy()
numeric_cols = [
  "age", "player_market_value", "cost", "latitude_to", "longitude_to",
  "latitude_from", "longitude_from"
]
for col in numeric_cols:
  transfers[col] = pd.to_numeric(transfers[col], errors="coerce")
transfers["date_of_transfer"] = pd.to_datetime(transfers["date_of_transfer"], errors="coerce")

paid = transfers.loc[transfers["cost"].gt(0)].copy()
paid = paid.dropna(subset=[
  "origin_club", "new_club", "league_origin_club", "league_new_club",
  "country_origin_club", "country_new_club", "cost"
])
paid["domestic"] = paid["country_origin_club"].eq(paid["country_new_club"])
paid["cross_league"] = ~paid["league_origin_club"].eq(paid["league_new_club"])
coord_cols = ["latitude_from", "longitude_from", "latitude_to", "longitude_to"]
paid_geo = paid.dropna(subset=coord_cols).copy()
paid_geo["distance_km"] = haversine_km(
  paid_geo["latitude_from"], paid_geo["longitude_from"],
  paid_geo["latitude_to"], paid_geo["longitude_to"]
)
paid_geo.loc[paid_geo["distance_km"].lt(1), "distance_km"] = 0

print(f"原始转会记录: {len(transfers_raw):,}")
print(f"付费转会记录: {len(paid):,}")
print(f"含完整坐标的付费转会: {len(paid_geo):,}")
print(f"国家数量: {paid['country_origin_club'].nunique()} -> {paid['country_new_club'].nunique()}")

原始转会记录: 19,139
付费转会记录: 1,326
含完整坐标的付费转会: 1,326
国家数量: 76 -> 64


## 2. 数据预处理

只保留 `cost > 0` 的实际付费转会；自由转会用于描述总体市场，但不进入定价和网络付费流动分析。

In [3]:
def latest_snapshot_csv(path, usecols, cache_name, chunksize=200_000):
  cache_path = CACHE_DIR / cache_name
  if cache_path.exists() and cache_path.stat().st_mtime > path.stat().st_mtime:
    return pd.read_pickle(cache_path)
  chunks = []
  latest_key = None
  for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
    if latest_key is None:
      latest_key = (chunk["fifa_version"].iloc[0], chunk["fifa_update"].iloc[0])
    mask = chunk["fifa_version"].eq(latest_key[0]) & chunk["fifa_update"].eq(latest_key[1])
    if mask.any():
      chunks.append(chunk.loc[mask].copy())
    if not mask.all():
      break
  result = pd.concat(chunks, ignore_index=True)
  result.to_pickle(cache_path)
  return result

player_cols = [
  "fifa_version", "fifa_update", "short_name", "long_name", "age", "overall", "potential",
  "value_eur", "league_name", "club_name", "pace", "shooting", "passing", "dribbling",
  "defending", "physic", "player_tags", "player_traits"
]
team_cols = [
  "fifa_version", "fifa_update", "team_name", "league_name", "overall", "attack",
  "midfield", "defence", "international_prestige", "domestic_prestige",
  "transfer_budget_eur", "club_worth_eur"
]
players = latest_snapshot_csv(PLAYERS_CSV, player_cols, "players_latest_network_geo.pkl")
teams = latest_snapshot_csv(TEAMS_CSV, team_cols, "teams_latest_network_geo.pkl")

print(f"最新球员快照: {players.shape}")
print(f"最新俱乐部快照: {teams.shape}")

最新球员快照: (18127, 18)
最新俱乐部快照: (712, 12)


## 3. 网络分析

In [4]:
def build_directed_graph(df, source_col, target_col, fee_col="cost"):
  graph = nx.DiGraph()
  grouped = df.groupby([source_col, target_col], as_index=False).agg(
    count=(fee_col, "size"),
    total_fee=(fee_col, "sum"),
    mean_fee=(fee_col, "mean")
  )
  for _, row in grouped.iterrows():
    if row[source_col] != row[target_col]:
      graph.add_edge(row[source_col], row[target_col],
                     count=float(row["count"]), total_fee=float(row["total_fee"]),
                     mean_fee=float(row["mean_fee"]), weight=float(row["count"]))
  return graph, grouped

club_graph, club_edges = build_directed_graph(paid, "origin_club", "new_club")
league_graph, league_edges = build_directed_graph(paid, "league_origin_club", "league_new_club")
country_graph, country_edges = build_directed_graph(paid, "country_origin_club", "country_new_club")

def centrality_table(graph):
  degree = nx.degree_centrality(graph)
  in_degree = dict(graph.in_degree(weight="count"))
  out_degree = dict(graph.out_degree(weight="count"))
  fee_in = dict(graph.in_degree(weight="total_fee"))
  fee_out = dict(graph.out_degree(weight="total_fee"))
  pagerank = nx.pagerank(graph, weight="total_fee")
  if graph.number_of_nodes() > 2:
    betweenness = nx.betweenness_centrality(graph, k=min(200, graph.number_of_nodes()), seed=RANDOM_STATE, weight=None)
  else:
    betweenness = {node: 0 for node in graph.nodes}
  rows = []
  for node in graph.nodes:
    rows.append({
      "node": node,
      "degree_centrality": degree.get(node, 0),
      "in_transfers": in_degree.get(node, 0),
      "out_transfers": out_degree.get(node, 0),
      "fee_in": fee_in.get(node, 0),
      "fee_out": fee_out.get(node, 0),
      "net_transfers": in_degree.get(node, 0) - out_degree.get(node, 0),
      "net_fee": fee_in.get(node, 0) - fee_out.get(node, 0),
      "pagerank": pagerank.get(node, 0),
      "betweenness": betweenness.get(node, 0),
    })
  return pd.DataFrame(rows)

club_cent = centrality_table(club_graph)
league_cent = centrality_table(league_graph)
country_cent = centrality_table(country_graph)

club_league = {}
for _, row in paid.iterrows():
  club_league.setdefault(row["origin_club"], row["league_origin_club"])
  club_league.setdefault(row["new_club"], row["league_new_club"])

print(f"俱乐部网络: {club_graph.number_of_nodes()} nodes, {club_graph.number_of_edges()} edges")
print(f"联赛网络: {league_graph.number_of_nodes()} nodes, {league_graph.number_of_edges()} edges")
print(f"国家网络: {country_graph.number_of_nodes()} nodes, {country_graph.number_of_edges()} edges")

俱乐部网络: 881 nodes, 1282 edges
联赛网络: 152 nodes, 594 edges
国家网络: 80 nodes, 435 edges


### 3.1 俱乐部转会网络

In [5]:
print("Generating network_01_club_transfer_network.png")
top_clubs = club_cent.assign(activity=lambda x: x["in_transfers"] + x["out_transfers"]).nlargest(80, "activity")["node"]
club_sub = club_graph.subgraph(top_clubs).copy()
pagerank_scores = club_cent.set_index("node")["pagerank"].to_dict()
pos = radial_layout(club_sub.nodes, pagerank_scores)
league_palette = {league: PALETTE[i % len(PALETTE)] for i, league in enumerate(sorted({club_league.get(n, "Unknown") for n in club_sub.nodes}))}
node_sizes = [180 + 15000 * club_cent.set_index("node").loc[n, "pagerank"] for n in club_sub.nodes]
node_colors = [league_palette.get(club_league.get(n, "Unknown"), MUTED) for n in club_sub.nodes]
edge_widths = [0.35 + 2.8 * math.log1p(club_sub[u][v].get("count", 1)) for u, v in club_sub.edges]

fig, ax = plt.subplots(figsize=(15, 10))
style_axis(ax, "Club Transfer Network: Top 80 Active Clubs", "节点大小=PageRank，颜色=主要联赛，边宽=付费转会次数")
for (u, v), width in zip(club_sub.edges, edge_widths):
  x0, y0 = pos[u]
  x1, y1 = pos[v]
  ax.plot([x0, x1], [y0, y1], color=MUTED, alpha=0.20, lw=width, zorder=1)
xy = np.array([pos[n] for n in club_sub.nodes])
ax.scatter(xy[:, 0], xy[:, 1], s=node_sizes, c=node_colors, alpha=0.92,
           edgecolors=BG, linewidths=0.6, zorder=2)
label_nodes = club_cent.nlargest(14, "pagerank")["node"].tolist()
for node in label_nodes:
  if node in pos:
    ax.text(pos[node][0], pos[node][1], clean_label(node, 18), fontsize=8.2, color=TEXT,
            ha="center", va="center", zorder=3,
            path_effects=[pe.withStroke(linewidth=3, foreground=BG)])
ax.set_axis_off()
savefig("network_01_club_transfer_network.png")

Generating network_01_club_transfer_network.png


Saved: charts/network_01_club_transfer_network.png


In [6]:
ranking = club_cent.nlargest(20, "degree_centrality").sort_values("degree_centrality", ascending=True)
fig, ax = plt.subplots(figsize=(10, 8))
style_axis(ax, "Top 20 Club Degree Centrality", "中心性越高，说明该俱乐部连接的买卖关系越广")
ax.barh(ranking["node"], ranking["degree_centrality"], color=CYAN, alpha=0.88)
ax.set_xlabel("Degree centrality")
ax.set_ylabel("")
savefig("network_02_club_centrality_ranking.png")

Saved: charts/network_02_club_centrality_ranking.png


In [7]:
buyer_seller = club_cent.copy()
buyer_seller["total_fee_m"] = euro_m(buyer_seller["fee_in"] + buyer_seller["fee_out"])
fig, ax = plt.subplots(figsize=(10.5, 8))
style_axis(ax, "Buyer vs Seller Clubs", "x=转出次数，y=转入次数；气泡大小=相关转会总额")
sizes = np.clip(buyer_seller["total_fee_m"], 1, 300) * 5
ax.scatter(buyer_seller["out_transfers"], buyer_seller["in_transfers"], s=sizes, color=GOLD, alpha=0.55, edgecolor=BG, linewidth=0.4)
max_axis = max(buyer_seller["out_transfers"].max(), buyer_seller["in_transfers"].max()) + 1
ax.plot([0, max_axis], [0, max_axis], color=MUTED, lw=1, ls="--", alpha=0.6)
add_labels(ax, buyer_seller, "out_transfers", "in_transfers", "node", n=10, dx=0.1, dy=0.1)
ax.set_xlabel("Outgoing paid transfers / seller activity")
ax.set_ylabel("Incoming paid transfers / buyer activity")
savefig("network_03_buyer_vs_seller_clubs.png")

Saved: charts/network_03_buyer_vs_seller_clubs.png


### 3.2 联赛间流动网络

In [8]:
league_flow = paid.loc[paid["league_origin_club"].ne(paid["league_new_club"])].groupby(
  ["league_origin_club", "league_new_club"], as_index=False
).agg(count=("cost", "size"), total_fee=("cost", "sum"))
top_league_nodes = pd.concat([
  league_flow.groupby("league_origin_club")["count"].sum(),
  league_flow.groupby("league_new_club")["count"].sum()
], axis=1).fillna(0).sum(axis=1).nlargest(14).index.tolist()
flow_top = league_flow[
  league_flow["league_origin_club"].isin(top_league_nodes) &
  league_flow["league_new_club"].isin(top_league_nodes)
].nlargest(36, "count")

left_order = flow_top.groupby("league_origin_club")["count"].sum().sort_values(ascending=False).index.tolist()
right_order = flow_top.groupby("league_new_club")["count"].sum().sort_values(ascending=False).index.tolist()
left_y = {label: i for i, label in enumerate(left_order)}
right_y = {label: i for i, label in enumerate(right_order)}

fig, ax = plt.subplots(figsize=(14, 9))
style_axis(ax, "League-to-League Transfer Flow", "Sankey-style: 左侧为输出联赛，右侧为输入联赛，线宽=转会次数")
for i, label in enumerate(left_order):
  ax.add_patch(Rectangle((0.02, i - 0.28), 0.02, 0.56, color=PALETTE[i % len(PALETTE)], alpha=0.9))
  ax.text(0.055, i, clean_label(label, 22), ha="left", va="center", fontsize=9, color=TEXT)
for i, label in enumerate(right_order):
  ax.add_patch(Rectangle((0.96, i - 0.28), 0.02, 0.56, color=PALETTE[i % len(PALETTE)], alpha=0.9))
  ax.text(0.945, i, clean_label(label, 22), ha="right", va="center", fontsize=9, color=TEXT)
max_count = flow_top["count"].max()
for idx, row in flow_top.iterrows():
  y0 = left_y[row["league_origin_club"]]
  y1 = right_y[row["league_new_club"]]
  width = 0.6 + 6.5 * row["count"] / max_count
  color = PALETTE[left_order.index(row["league_origin_club"]) % len(PALETTE)]
  draw_curve(ax, 0.12, y0, 0.88, y1, color=color, width=width, alpha=0.35, curve=0.12)
ax.set_xlim(0, 1)
ax.set_ylim(max(len(left_order), len(right_order)) - 0.2, -0.8)
ax.set_xticks([])
ax.set_yticks([])
savefig("network_04_league_flow_sankey.png")

Saved: charts/network_04_league_flow_sankey.png


In [9]:
league_balance = league_cent.assign(net_count=lambda x: x["in_transfers"] - x["out_transfers"])
league_balance_top = pd.concat([
  league_balance.nlargest(12, "net_count"),
  league_balance.nsmallest(12, "net_count")
]).drop_duplicates("node").sort_values("net_count")
fig, ax = plt.subplots(figsize=(11, 8))
style_axis(ax, "League Transfer Balance", "正值=净流入，负值=净流出；按付费转会次数计算")
colors = [GREEN if x >= 0 else RED for x in league_balance_top["net_count"]]
ax.barh(league_balance_top["node"], league_balance_top["net_count"], color=colors, alpha=0.88)
ax.axvline(0, color=MUTED, lw=1)
ax.set_xlabel("Incoming paid transfers - outgoing paid transfers")
ax.set_ylabel("")
savefig("network_05_league_balance.png")

Saved: charts/network_05_league_balance.png


In [10]:
major_leagues = ["Premier League", "LaLiga", "Serie A", "Bundesliga", "Ligue 1"]
major_edges = league_flow[
  league_flow["league_origin_club"].isin(major_leagues) &
  league_flow["league_new_club"].isin(major_leagues)
].copy()
major_graph = nx.DiGraph()
for _, row in major_edges.iterrows():
  major_graph.add_edge(row["league_origin_club"], row["league_new_club"], count=row["count"], total_fee=row["total_fee"])
pos_major = nx.circular_layout(major_graph)
fig, ax = plt.subplots(figsize=(9, 8))
style_axis(ax, "Big Five League Transfer Connections", "五大联赛之间的付费转会网络，边宽=次数")
widths = [0.8 + 5 * major_graph[u][v]["count"] / max(1, major_edges["count"].max()) for u, v in major_graph.edges]
nx.draw_networkx_edges(major_graph, pos_major, width=widths, edge_color=CYAN, alpha=0.42, arrows=True, arrowsize=18, connectionstyle="arc3,rad=0.16", ax=ax)
nx.draw_networkx_nodes(major_graph, pos_major, node_size=2600, node_color=PURPLE, edgecolors=BG, linewidths=1.2, ax=ax)
nx.draw_networkx_labels(major_graph, pos_major, font_size=10, font_color=TEXT, ax=ax)
edge_labels = {(u, v): int(d["count"]) for u, v, d in major_graph.edges(data=True)}
nx.draw_networkx_edge_labels(major_graph, pos_major, edge_labels=edge_labels, font_size=8, font_color=MUTED, ax=ax)
ax.set_axis_off()
savefig("network_06_top_league_connections.png")

Saved: charts/network_06_top_league_connections.png


### 3.3 国家转会网络

In [11]:
top_countries = country_cent.assign(activity=lambda x: x["in_transfers"] + x["out_transfers"]).nlargest(55, "activity")["node"]
country_sub = country_graph.subgraph(top_countries).copy()
country_pagerank_scores = country_cent.set_index("node")["pagerank"].to_dict()
pos_country = radial_layout(country_sub.nodes, country_pagerank_scores)
country_sizes = [250 + 11000 * country_cent.set_index("node").loc[n, "pagerank"] for n in country_sub.nodes]
fig, ax = plt.subplots(figsize=(13, 10))
style_axis(ax, "Country Transfer Network", "节点大小=PageRank，边宽=国家间付费转会次数")
edge_widths = [0.4 + 3.5 * math.log1p(country_sub[u][v].get("count", 1)) for u, v in country_sub.edges]
for (u, v), width in zip(country_sub.edges, edge_widths):
  x0, y0 = pos_country[u]
  x1, y1 = pos_country[v]
  ax.plot([x0, x1], [y0, y1], color=MUTED, alpha=0.22, lw=width, zorder=1)
country_xy = np.array([pos_country[n] for n in country_sub.nodes])
ax.scatter(country_xy[:, 0], country_xy[:, 1], s=country_sizes, color=GREEN,
           alpha=0.88, linewidths=0.7, edgecolors=BG, zorder=2)
label_countries = country_cent.nlargest(16, "pagerank")["node"].tolist()
for node in label_countries:
  if node in pos_country:
    ax.text(pos_country[node][0], pos_country[node][1], node, fontsize=8.5, color=TEXT,
            ha="center", va="center", zorder=3,
            path_effects=[pe.withStroke(linewidth=3, foreground=BG)])
ax.set_axis_off()
savefig("network_07_country_transfer_network.png")

Saved: charts/network_07_country_transfer_network.png


In [12]:
COUNTRY_CONTINENT = {
  "England": "Europe", "Spain": "Europe", "Italy": "Europe", "Germany": "Europe", "France": "Europe",
  "Portugal": "Europe", "Netherlands": "Europe", "Belgium": "Europe", "Turkey": "Europe", "Russia": "Europe",
  "Scotland": "Europe", "Austria": "Europe", "Switzerland": "Europe", "Denmark": "Europe", "Sweden": "Europe",
  "Norway": "Europe", "Poland": "Europe", "Romania": "Europe", "Greece": "Europe", "Czech Republic": "Europe",
  "Croatia": "Europe", "Serbia": "Europe", "Ukraine": "Europe", "Hungary": "Europe", "Cyprus": "Europe",
  "Slovenia": "Europe", "Slovakia": "Europe", "Bulgaria": "Europe", "Finland": "Europe", "Ireland": "Europe",
  "Brazil": "South America", "Argentina": "South America", "Uruguay": "South America", "Colombia": "South America",
  "Chile": "South America", "Ecuador": "South America", "Paraguay": "South America", "Peru": "South America",
  "Venezuela": "South America", "Bolivia": "South America",
  "United States": "North America", "Mexico": "North America", "Canada": "North America", "Costa Rica": "North America",
  "China": "Asia", "Japan": "Asia", "Korea, South": "Asia", "Saudi Arabia": "Asia", "Qatar": "Asia",
  "United Arab Emirates": "Asia", "India": "Asia", "Thailand": "Asia", "Australia": "Oceania",
  "South Africa": "Africa", "Egypt": "Africa", "Morocco": "Africa", "Tunisia": "Africa", "Algeria": "Africa",
}
paid["continent_origin"] = paid["country_origin_club"].map(COUNTRY_CONTINENT).fillna("Other")
paid["continent_new"] = paid["country_new_club"].map(COUNTRY_CONTINENT).fillna("Other")
continent_flow = paid.groupby(["continent_origin", "continent_new"], as_index=False).agg(
  count=("cost", "size"),
  median_fee=("cost", "median"),
  total_fee=("cost", "sum")
)
continent_matrix = continent_flow.pivot(index="continent_origin", columns="continent_new", values="count").fillna(0)
fig, ax = plt.subplots(figsize=(9.5, 7.5))
style_axis(ax, "Inter-continental Transfer Flow", "数值为付费转会次数；颜色越亮，区域连接越强")
sns.heatmap(continent_matrix, cmap="mako", annot=True, fmt=".0f", linewidths=0.5, linecolor=GRID,
            cbar_kws={"label": "Paid transfer count"}, ax=ax)
ax.set_xlabel("Destination continent")
ax.set_ylabel("Origin continent")
savefig("network_08_continent_flow.png")

Saved: charts/network_08_continent_flow.png


## 4. 地理数据可视化

In [13]:
country_flow_geo = paid_geo.groupby(
  ["country_origin_club", "country_new_club"], as_index=False
).agg(
  count=("cost", "size"),
  total_fee=("cost", "sum"),
  lat_from=("latitude_from", "median"),
  lon_from=("longitude_from", "median"),
  lat_to=("latitude_to", "median"),
  lon_to=("longitude_to", "median"),
)
country_flow_geo = country_flow_geo.loc[country_flow_geo["country_origin_club"].ne(country_flow_geo["country_new_club"])]
country_flow_geo = country_flow_geo.nlargest(120, "total_fee")

fig, ax = plt.subplots(figsize=(16, 8.5))
style_axis(ax, "Global Transfer Flow Map", "经纬度流向图；线宽=转会费总额，节点=国家中心坐标")
ax.set_xlim(-170, 180)
ax.set_ylim(-58, 75)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(np.arange(-150, 181, 50))
ax.set_yticks(np.arange(-50, 76, 25))
max_fee = country_flow_geo["total_fee"].max()
for _, row in country_flow_geo.iterrows():
  width = 0.25 + 4.8 * row["total_fee"] / max_fee
  color = GREEN if row["country_origin_club"] == row["country_new_club"] else CYAN
  draw_curve(ax, row["lon_from"], row["lat_from"], row["lon_to"], row["lat_to"], color=color, width=width, alpha=0.28, curve=0.04)
ax.scatter(country_flow_geo["lon_from"], country_flow_geo["lat_from"], s=18, color=ORANGE, alpha=0.65, label="Origin")
ax.scatter(country_flow_geo["lon_to"], country_flow_geo["lat_to"], s=24, color=CYAN, alpha=0.75, label="Destination")
top_geo_labels = pd.concat([
  country_flow_geo.nlargest(8, "total_fee")[["country_origin_club", "lon_from", "lat_from"]].rename(columns={"country_origin_club": "country", "lon_from": "lon", "lat_from": "lat"}),
  country_flow_geo.nlargest(8, "total_fee")[["country_new_club", "lon_to", "lat_to"]].rename(columns={"country_new_club": "country", "lon_to": "lon", "lat_to": "lat"})
]).drop_duplicates("country")
for _, row in top_geo_labels.iterrows():
  ax.text(row["lon"] + 2, row["lat"] + 1, row["country"], fontsize=8.5, color=TEXT,
          path_effects=[pe.withStroke(linewidth=3, foreground=BG)])
ax.legend(loc="lower left")
savefig("geo_01_global_transfer_flow_map.png")

Saved: charts/geo_01_global_transfer_flow_map.png


In [14]:
europe_geo = paid_geo[
  paid_geo["longitude_to"].between(-25, 45) &
  paid_geo["latitude_to"].between(34, 72)
].copy()
fig, ax = plt.subplots(figsize=(10, 9))
style_axis(ax, "Europe Transfer Destination Heatmap", "按转入地坐标统计付费转会费密度")
hb = ax.hexbin(europe_geo["longitude_to"], europe_geo["latitude_to"], C=europe_geo["cost"], gridsize=34,
               reduce_C_function=np.sum, cmap="rocket", mincnt=1, alpha=0.88)
cb = fig.colorbar(hb, ax=ax)
cb.set_label("Total transfer fee")
cb.ax.yaxis.label.set_color(TEXT)
cb.ax.tick_params(colors=TEXT)
ax.scatter(europe_geo["longitude_to"], europe_geo["latitude_to"], s=8, color=CYAN, alpha=0.14)
ax.set_xlim(-15, 38)
ax.set_ylim(35, 62)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
savefig("geo_02_europe_transfer_heatmap.png")

Saved: charts/geo_02_europe_transfer_heatmap.png


In [15]:
geo_sample = paid_geo.loc[paid_geo["distance_km"].gt(0)].copy()
geo_sample["log_fee"] = np.log10(geo_sample["cost"])
geo_sample["log_distance"] = np.log10(geo_sample["distance_km"] + 1)
fig, ax = plt.subplots(figsize=(10.5, 8))
style_axis(ax, "Geographic Distance vs Transfer Fee", "横轴为地理距离，纵轴为实际转会费；两轴使用 log10")
sns.regplot(data=geo_sample, x="log_distance", y="log_fee", scatter_kws={"s": 28, "alpha": 0.42, "color": CYAN},
            line_kws={"color": GOLD, "lw": 2.2}, ax=ax)
ax.set_xlabel("log10(distance km + 1)")
ax.set_ylabel("log10(transfer fee EUR)")
savefig("geo_03_distance_vs_fee.png")

Saved: charts/geo_03_distance_vs_fee.png


In [16]:
box_df = paid_geo.copy()
box_df["transfer_scope"] = np.where(box_df["domestic"], "Domestic", "International")
box_sample = box_df.sample(min(700, len(box_df)), random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(8.5, 7))
style_axis(ax, "Domestic vs International Transfer Price", "箱线图展示价格分布，纵轴使用 log10")
sns.boxplot(data=box_df, x="transfer_scope", y=np.log10(box_df["cost"]), palette=[GREEN, PURPLE], ax=ax)
sns.stripplot(data=box_sample,
              x="transfer_scope", y=np.log10(box_sample["cost"]),
              color=TEXT, alpha=0.18, size=2, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("log10(transfer fee EUR)")
savefig("geo_04_domestic_vs_international.png")

Saved: charts/geo_04_domestic_vs_international.png


In [17]:
def route_type(row):
  origin = row["continent_origin"]
  dest = row["continent_new"]
  if origin == "Europe" and dest == "Europe":
    return "Europe internal"
  if origin == "South America" and dest == "Europe":
    return "South America -> Europe"
  if origin == "Africa" and dest == "Europe":
    return "Africa -> Europe"
  if dest == "Europe" and origin != "Europe":
    return "Other -> Europe"
  if origin == dest:
    return f"{origin} internal"
  return "Other cross-region"

paid["route_type"] = paid.apply(route_type, axis=1)
route_stats = paid.groupby("route_type", as_index=False).agg(
  count=("cost", "size"),
  median_fee=("cost", "median"),
  mean_fee=("cost", "mean")
).sort_values("median_fee", ascending=False)
fig, ax1 = plt.subplots(figsize=(12, 7.5))
style_axis(ax1, "Regional Transfer Patterns", "柱形=中位转会费，折线=转会次数")
route_plot = route_stats.head(10).copy()
bars = ax1.bar(route_plot["route_type"], euro_m(route_plot["median_fee"]), color=ORANGE, alpha=0.88)
ax1.set_ylabel("Median fee (€m)")
ax1.tick_params(axis="x", rotation=28)
ax2 = ax1.twinx()
ax2.plot(route_plot["route_type"], route_plot["count"], color=CYAN, marker="o", lw=2.4)
ax2.set_ylabel("Paid transfer count", color=TEXT)
ax2.tick_params(colors=TEXT)
for spine in ax2.spines.values():
  spine.set_color(GRID)
savefig("geo_05_regional_transfer_patterns.png")

Saved: charts/geo_05_regional_transfer_patterns.png


## 5. 文本分析

In [18]:
players_text = players.copy()
players_text["name_key_short"] = players_text["short_name"].map(normalize_name)
players_text["name_key_long"] = players_text["long_name"].map(normalize_name)
short_map = players_text.sort_values("value_eur", ascending=False).drop_duplicates("name_key_short")
long_map = players_text.sort_values("value_eur", ascending=False).drop_duplicates("name_key_long")

paid_text = paid.copy().reset_index(drop=True)
paid_text["name_key"] = paid_text["name"].map(normalize_name)
merged = paid_text.merge(
  short_map.drop(columns=["name_key_long"]).rename(columns={"name_key_short": "name_key"}),
  on="name_key", how="left", suffixes=("", "_player")
)
missing = merged["player_tags"].isna().to_numpy()
fallback = paid_text.loc[missing].merge(
  long_map.drop(columns=["name_key_short"]).rename(columns={"name_key_long": "name_key"}),
  on="name_key", how="left", suffixes=("", "_player")
)
for col in fallback.columns:
  if col in merged.columns:
    merged.loc[missing, col] = fallback[col].values

def split_tags(value):
  if pd.isna(value) or not str(value).strip():
    return []
  tags = [re.sub(r"^#", "", item.strip()) for item in str(value).split(",")]
  return [tag for tag in tags if tag]

merged["tag_list"] = merged["player_tags"].apply(split_tags)
matched_text = merged.loc[merged["tag_list"].map(len).gt(0)].copy()
high_threshold = paid["cost"].quantile(0.90)
high_value_tags = matched_text.loc[matched_text["cost"].ge(high_threshold), "tag_list"].sum()
tag_counter = Counter(high_value_tags)
print(f"文本匹配到标签的付费转会: {len(matched_text):,}")
print(f"高价阈值(top 10%): €{high_threshold:,.0f}")
print(tag_counter.most_common(10))

文本匹配到标签的付费转会: 85
高价阈值(top 10%): €11,000,000
[('Acrobat', 7), ('Strength', 6), ('Engine', 6), ('Dribbler', 6), ('Speedster', 5), ('Aerial Threat', 4), ('Tackling', 3), ('Tactician', 2), ('Crosser', 2), ('Distance Shooter', 2)]


In [19]:
word_freq = dict(tag_counter)
if not word_freq:
  word_freq = {"No matched tags": 1}
wordcloud = WordCloud(
  width=1400, height=800, background_color=BG, colormap="cool",
  prefer_horizontal=0.9, random_state=RANDOM_STATE, max_words=80
).generate_from_frequencies(word_freq)
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(wordcloud, interpolation="bilinear")
ax.set_title("High-value Player Tags Wordcloud", loc="left", color=TEXT, fontsize=18, fontweight="bold")
ax.axis("off")
savefig("text_01_high_value_player_tags_wordcloud.png")

Saved: charts/text_01_high_value_player_tags_wordcloud.png


In [20]:
tag_rows = []
for _, row in matched_text.iterrows():
  for tag in row["tag_list"]:
    tag_rows.append({"tag": tag, "cost": row["cost"], "club": row["new_club"]})
tag_df = pd.DataFrame(tag_rows)
tag_price = tag_df.groupby("tag", as_index=False).agg(
  avg_fee=("cost", "mean"),
  median_fee=("cost", "median"),
  count=("cost", "size")
)
tag_price_top = tag_price.loc[tag_price["count"].ge(2)].nlargest(10, "avg_fee").sort_values("avg_fee")
fig, ax = plt.subplots(figsize=(10.5, 7.5))
style_axis(ax, "Tags Associated with Higher Transfer Fees", "至少出现 2 次；按平均转会费排序")
ax.barh(tag_price_top["tag"], euro_m(tag_price_top["avg_fee"]), color=PINK, alpha=0.9)
ax.set_xlabel("Average fee (€m)")
ax.set_ylabel("")
savefig("text_02_tags_vs_price.png")

Saved: charts/text_02_tags_vs_price.png


In [21]:
central_clubs = set(club_cent.nlargest(25, "pagerank")["node"])
central_tag_df = tag_df.loc[tag_df["club"].isin(central_clubs)].copy()
central_tags = central_tag_df["tag"].value_counts().head(12).sort_values()
fig, ax = plt.subplots(figsize=(10, 7.5))
style_axis(ax, "Preferred Tags Bought by Central Clubs", "中心俱乐部按 PageRank 前 25 定义；柱形=购买球员标签出现次数")
ax.barh(central_tags.index, central_tags.values, color=GREEN, alpha=0.9)
ax.set_xlabel("Tag frequency among central-club purchases")
ax.set_ylabel("")
savefig("text_03_central_clubs_preferred_tags.png")

Saved: charts/text_03_central_clubs_preferred_tags.png


## 6. 综合分析

In [22]:
league_inbound_price = paid.groupby("league_new_club", as_index=False).agg(
  avg_fee=("cost", "mean"),
  median_fee=("cost", "median"),
  transfer_count=("cost", "size")
)
league_integrated = league_cent.rename(columns={"node": "league"}).merge(
  league_inbound_price, left_on="league", right_on="league_new_club", how="left"
).dropna(subset=["avg_fee"])
fig, ax = plt.subplots(figsize=(10.5, 8))
style_axis(ax, "League Network Centrality vs Average Transfer Price", "气泡大小=转入付费转会次数；中心性用 PageRank 衡量")
sizes = 30 + league_integrated["transfer_count"] * 12
ax.scatter(league_integrated["pagerank"], euro_m(league_integrated["avg_fee"]), s=sizes, color=CYAN, alpha=0.68, edgecolor=BG, linewidth=0.6)
top_label = league_integrated.nlargest(12, "avg_fee")
for _, row in top_label.iterrows():
  ax.text(row["pagerank"] + 0.0005, euro_m(row["avg_fee"]) + 0.08, clean_label(row["league"], 18), fontsize=8.5, color=TEXT,
          path_effects=[pe.withStroke(linewidth=3, foreground=BG)])
ax.set_xlabel("League PageRank in transfer network")
ax.set_ylabel("Average inbound transfer fee (€m)")
savefig("integrated_01_league_centrality_vs_price.png")

Saved: charts/integrated_01_league_centrality_vs_price.png


In [23]:
route_premium = paid.groupby("route_type", as_index=False).agg(
  median_fee=("cost", "median"),
  mean_fee=("cost", "mean"),
  count=("cost", "size")
)
baseline = route_premium.loc[route_premium["route_type"].eq("Europe internal"), "median_fee"]
baseline_value = baseline.iloc[0] if len(baseline) else paid["cost"].median()
route_premium["premium_vs_europe_internal"] = route_premium["median_fee"] / baseline_value - 1
route_premium_plot = route_premium.loc[route_premium["count"].ge(5)].sort_values("premium_vs_europe_internal")
fig, ax = plt.subplots(figsize=(11, 7.5))
style_axis(ax, "Transfer Path Premium", "相对欧洲内部转会中位价的溢价；仅显示样本数 >= 5 的路径")
colors = [GREEN if x >= 0 else RED for x in route_premium_plot["premium_vs_europe_internal"]]
ax.barh(route_premium_plot["route_type"], route_premium_plot["premium_vs_europe_internal"] * 100, color=colors, alpha=0.88)
ax.axvline(0, color=MUTED, lw=1)
ax.set_xlabel("Median fee premium vs Europe internal (%)")
ax.set_ylabel("")
savefig("integrated_02_transfer_path_premium.png")

Saved: charts/integrated_02_transfer_path_premium.png


## 7. 结果汇总

In [24]:
summary = {
  "records_total": int(len(transfers_raw)),
  "records_paid": int(len(paid)),
  "records_paid_geo": int(len(paid_geo)),
  "clubs": int(club_graph.number_of_nodes()),
  "club_edges": int(club_graph.number_of_edges()),
  "leagues": int(league_graph.number_of_nodes()),
  "countries": int(country_graph.number_of_nodes()),
  "top_club_pagerank": club_cent.nlargest(1, "pagerank").iloc[0]["node"],
  "top_club_degree": club_cent.nlargest(1, "degree_centrality").iloc[0]["node"],
  "top_buyer_club": club_cent.nlargest(1, "in_transfers").iloc[0]["node"],
  "top_seller_club": club_cent.nlargest(1, "out_transfers").iloc[0]["node"],
  "top_net_in_league": league_balance.nlargest(1, "net_count").iloc[0]["node"],
  "top_net_out_league": league_balance.nsmallest(1, "net_count").iloc[0]["node"],
  "median_domestic_fee": float(box_df.loc[box_df["domestic"], "cost"].median()),
  "median_international_fee": float(box_df.loc[~box_df["domestic"], "cost"].median()),
  "distance_fee_corr": float(geo_sample[["log_distance", "log_fee"]].corr().iloc[0, 1]),
  "top_high_value_tag": tag_counter.most_common(1)[0][0] if tag_counter else "N/A",
  "text_matched_records": int(len(matched_text)),
}
pd.Series(summary).to_json(OUTPUT_DIR / "network_geo_summary.json", force_ascii=False, indent=2)
club_cent.to_csv(OUTPUT_DIR / "club_network_centrality.csv", index=False)
league_cent.to_csv(OUTPUT_DIR / "league_network_centrality.csv", index=False)
route_premium.to_csv(OUTPUT_DIR / "transfer_path_premium.csv", index=False)

print(pd.Series(summary).to_string())
print("All charts saved in charts/.")

records_total                           19139
records_paid                             1326
records_paid_geo                         1326
clubs                                     881
club_edges                               1282
leagues                                   152
countries                                  80
top_club_pagerank                   Fulham FC
top_club_degree                   Sporting CP
top_buyer_club              Nottingham Forest
top_seller_club                       AS Roma
top_net_in_league              Premier League
top_net_out_league              Liga Portugal
median_domestic_fee                  700000.0
median_international_fee            1820000.0
distance_fee_corr                   -0.065333
top_high_value_tag                    Acrobat
text_matched_records                       85
All charts saved in charts/.
